# Block 1 — Tracker State and Straight-Line Projection

The AMS-02 silicon tracker reconstructs the trajectory of a charged particle before it reaches the Electromagnetic Calorimeter (ECAL). To analyze the energy deposited in the ECAL relative to the incoming particle, we must project the reconstructed track to the center of each ECAL layer.

In this notebook, we will:

1. represent a reconstructed track using a reference position and two direction angles;
2. derive the equations for straight-line propagation;
3. project an example track to the 18 ECAL layer centers;
4. verify the special case of a vertical track.

This block works entirely in continuous spatial coordinates. It does **not** yet assign projected positions to ECAL cells or account for alternating readout directions. Those operations belong to Block 2.

## 1. Coordinate and angle conventions

We use the local ECAL coordinate system established in Block 0:

- $x$ and $y$ describe transverse positions across the ECAL face;
- $z$ increases through the ECAL depth;
- all positions and distances are measured in millimetres;
- all angles are measured in radians in the implementation.

A reconstructed track is defined at a reference position

$$
\mathbf{r}_0 =
\begin{pmatrix}
x_0 \\
y_0 \\
z_0
\end{pmatrix}.
$$

Its direction is described by:

- $\theta$: the polar angle measured from the positive $z$-axis;
- $\phi$: the azimuthal angle measured from the positive $x$-axis toward the positive $y$-axis.

The corresponding unit direction vector is

$$
\hat{\mathbf{u}}
=
\begin{pmatrix}
\sin\theta\cos\phi \\
\sin\theta\sin\phi \\
\cos\theta
\end{pmatrix}.
$$

Therefore, the track can be written parametrically as

$$
\mathbf{r}(s)=\mathbf{r}_0+s\hat{\mathbf{u}},
$$

where $s$ is the signed distance travelled along the track.

## 2. Projection to a known z-coordinate

Suppose an ECAL layer is centered at $z=z_\ell$. From the $z$-component of the parametric track equation,

$$
z_\ell=z_0+s_\ell\cos\theta.
$$

Solving for the distance parameter gives

$$
s_\ell=\frac{z_\ell-z_0}{\cos\theta}.
$$

Substituting this result into the $x$- and $y$-components gives

$$
x_\ell
=
x_0+(z_\ell-z_0)\tan\theta\cos\phi,
$$

and

$$
y_\ell
=
y_0+(z_\ell-z_0)\tan\theta\sin\phi.
$$

It is useful to define the transverse slopes with respect to $z$:

$$
t_x=\frac{dx}{dz}=\tan\theta\cos\phi,
$$

$$
t_y=\frac{dy}{dz}=\tan\theta\sin\phi.
$$

The projection equations then become

$$
x_\ell=x_0+(z_\ell-z_0)t_x,
$$

$$
y_\ell=y_0+(z_\ell-z_0)t_y.
$$

These equations assume that the reconstructed trajectory can be propagated as a straight line between the reference point and the ECAL. Effects such as multiple scattering, magnetic curvature, track-fit uncertainty, and shower development are outside the scope of this block.

In [1]:
from math import cos, radians, sin, tan
from pathlib import Path

import pytest

from ams_ecal import load_geometry


def find_repository_root(start: Path) -> Path:
    """Find the repository root from Jupyter's current directory."""

    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate

    raise FileNotFoundError("Could not locate the repository root.")


repository_root = find_repository_root(Path.cwd())
geometry = load_geometry(repository_root / "configs" / "geometry.yaml")

len(geometry.uniform_layer_centers_z_mm)

18

## 3. Numerical projection example

We now define an example reconstructed track with reference position

$$
(x_0,y_0,z_0)=(10,-20,0)\ \text{mm},
$$

and direction angles

$$
\theta=10^\circ,
\qquad
\phi=30^\circ.
$$

Although the angles are easier to describe in degrees, Python's trigonometric functions expect radians. We therefore convert them using `radians()`.

The transverse slopes $t_x$ and $t_y$ describe how much the track's $x$- and $y$-coordinates change for every millimetre travelled in the positive $z$-direction.

In [2]:
# Example reconstructed track state in the local ECAL coordinate system.
x0_mm = 10.0
y0_mm = -20.0
z0_mm = 0.0

theta_rad = radians(10.0)
phi_rad = radians(30.0)

slope_x = tan(theta_rad) * cos(phi_rad)
slope_y = tan(theta_rad) * sin(phi_rad)

slope_x, slope_y

(0.1527036446661393, 0.08816349035423247)

Because both slopes are positive, the projected $x$- and $y$-coordinates should increase as the track passes deeper into the ECAL.

The reference value $y_0=-20\ \text{mm}$ is negative, so an increasing $y$-coordinate means that it becomes less negative before potentially crossing zero.

We will evaluate the projection equations at every layer-center $z$-coordinate supplied by the Block 0 geometry model.

In [3]:
projected_points = tuple(
    (
        x0_mm + (z_mm - z0_mm) * slope_x,
        y0_mm + (z_mm - z0_mm) * slope_y,
        z_mm,
    )
    for z_mm in geometry.uniform_layer_centers_z_mm
)

projected_points[0], projected_points[-1]

((10.706254356580894, -19.592243857111676, 4.625),
 (34.7189024803313, -5.728534998908618, 161.875))

## 4. Interpreting the result

The first projected point lies at the center of the first ECAL layer, while the last lies at the center of the eighteenth layer.

For this example:

- the $x$-coordinate increases from approximately $10.71\ \text{mm}$ to $34.72\ \text{mm}$;
- the $y$-coordinate increases from approximately $-19.59\ \text{mm}$ to $-5.73\ \text{mm}$;
- the $z$-coordinates come directly from the detector geometry.

The projection currently returns continuous physical coordinates. For example, $x=10.71\ \text{mm}$ has not yet been converted into an integer ECAL cell index.

Keeping continuous projection and discrete cell mapping separate lets us test the track geometry independently of the detector readout mapping.

## 5. Vertical-track limit

A useful physical and numerical check is the vertical-track case.

If

$$
\theta=0,
$$

then

$$
\tan\theta=0.
$$

Consequently,

$$
t_x=t_y=0,
$$

regardless of the value of $\phi$. The projection equations reduce to

$$
x_\ell=x_0,
\qquad
y_\ell=y_0.
$$

A vertical track should therefore retain the same transverse coordinates at every ECAL layer.

In [4]:
vertical_theta_rad = 0.0

vertical_points = tuple(
    (
        x0_mm
        + (z_mm - z0_mm)
        * tan(vertical_theta_rad)
        * cos(phi_rad),
        y0_mm
        + (z_mm - z0_mm)
        * tan(vertical_theta_rad)
        * sin(phi_rad),
        z_mm,
    )
    for z_mm in geometry.uniform_layer_centers_z_mm
)

assert all(x_mm == x0_mm for x_mm, _, _ in vertical_points)
assert all(y_mm == y0_mm for _, y_mm, _ in vertical_points)

vertical_points[0], vertical_points[-1]

((10.0, -20.0, 4.625), (10.0, -20.0, 161.875))

## Checkpoint conclusions

We have derived and numerically verified the straight-line track-projection model:

$$
x(z)=x_0+(z-z_0)\tan\theta\cos\phi,
$$

$$
y(z)=y_0+(z-z_0)\tan\theta\sin\phi.
$$

The numerical example showed that an inclined track changes its transverse position as it travels through the ECAL. The vertical-track test confirmed that setting $\theta=0$ preserves $x_0$ and $y_0$ at every layer.

At this checkpoint, the equations exist only as exploratory notebook code. The next step is to move the physical track state into an immutable `TrackState` class and move the projection calculation into reusable, testable functions.

Cell indexing, alternating ECAL readout views, and shower-centered cropping remain outside Block 1.

## 6. Representing a reconstructed track as an object

Our exploratory calculation stored the track state in five independent variables:

- `x0_mm`;
- `y0_mm`;
- `z0_mm`;
- `theta_rad`;
- `phi_rad`.

These values describe one physical entity and must remain consistent with one another. We therefore group them in a `TrackState` class.

The position

$$
\mathbf{r}_0=(x_0,y_0,z_0)
$$

specifies where the reconstructed track is defined. This reference point does not have to lie inside the ECAL. It could, for example, lie on an upstream tracker plane.

The angles describe a track directed into the ECAL:

$$
0\leq\theta<\frac{\pi}{2},
$$

and

$$
0\leq\phi<2\pi.
$$

The upper limit $\theta=\pi/2$ is excluded because it would describe a track parallel to the ECAL layers. Such a track has no positive $z$ component and cannot be projected to deeper layer-center $z$ coordinates using our equations.

### Why make the track state immutable?

A reconstructed track is an input to the ECAL projection. Once created, its fitted position and direction should not change accidentally while another calculation is using it.

The `TrackState` class is therefore defined using

```python
@dataclass(frozen=True, slots=True)

In [5]:
from ams_ecal import TrackState

track = TrackState(
    x0_mm=10.0,
    y0_mm=-20.0,
    z0_mm=0.0,
    theta_rad=radians(10.0),
    phi_rad=radians(30.0),
)

track

TrackState(x0_mm=10.0, y0_mm=-20.0, z0_mm=0.0, theta_rad=0.17453292519943295, phi_rad=0.5235987755982988)

## 7. Derived direction quantities

The reference position and angles are the stored state. The Cartesian direction components are derived from those angles:

$$
u_x=\sin\theta\cos\phi,
$$

$$
u_y=\sin\theta\sin\phi,
$$

$$
u_z=\cos\theta.
$$

The slopes with respect to $z$ are then derived from the direction components:

$$
t_x=\frac{u_x}{u_z},
\qquad
t_y=\frac{u_y}{u_z}.
$$

This is preferable to storing the direction vector and slopes separately. If all of them were independently stored, they could disagree. Deriving them from one authoritative pair of angles guarantees internal consistency.

In [6]:
direction = track.direction_unit_vector
track_slopes = track.slopes_wrt_z

direction, track_slopes

((0.1503837331804353, 0.08682408883346515, 0.984807753012208),
 (0.1527036446661393, 0.08816349035423247))

The direction components and slopes are different quantities.

The unit-vector components describe displacement per unit distance travelled along the track:

$$
u_x=\frac{dx}{ds},
\qquad
u_y=\frac{dy}{ds},
\qquad
u_z=\frac{dz}{ds}.
$$

The transverse slopes describe displacement per unit increase in ECAL depth:

$$
t_x=\frac{dx}{dz},
\qquad
t_y=\frac{dy}{dz}.
$$

Because

$$
\frac{dx}{dz}
=
\frac{dx/ds}{dz/ds},
$$

we obtain

$$
t_x=\frac{u_x}{u_z},
\qquad
t_y=\frac{u_y}{u_z}.
$$

For an inclined track, one millimetre of distance along the track produces less than one millimetre of $z$ displacement. This is why the numerical transverse slopes are slightly larger than the corresponding $x$ and $y$ unit-vector components.

In [7]:
from math import sqrt

direction_magnitude = sqrt(
    sum(component**2 for component in direction)
)

assert direction_magnitude == pytest.approx(1.0)
assert track_slopes == pytest.approx((slope_x, slope_y))

direction_magnitude

0.9999999999999999

## Track-state checkpoint conclusions

We have replaced five independent track variables with one validated and immutable `TrackState`.

The object stores only the reconstructed reference position and direction angles. Quantities that follow mathematically from those angles—such as the Cartesian unit vector and transverse slopes—are calculated as derived properties.

This gives us one consistent representation:

$$
\text{stored track state}
\longrightarrow
\hat{\mathbf{u}}
\longrightarrow
(t_x,t_y).
$$

The numerical results agree with the exploratory calculations from the first checkpoint.

The next checkpoint will implement a pure function that accepts a `TrackState` and a target $z$ coordinate and returns the projected continuous point

$$
(x(z),y(z),z).
$$

Cell indices and alternating ECAL readout coordinates remain outside Block 1.

## 8. Reusable straight-line projection

We can now move the exploratory projection equation into a reusable pure function.

The function accepts:

1. an immutable `TrackState`;
2. a target depth `target_z_mm`.

It returns the continuous point

$$
(x(z),y(z),z).
$$

Starting from the transverse slopes

$$
t_x=\frac{dx}{dz},
\qquad
t_y=\frac{dy}{dz},
$$

the displacement from the reference plane to the target plane is

$$
\Delta z=z-z_0.
$$

The projected coordinates are therefore

$$
x(z)=x_0+\Delta z\,t_x,
$$

and

$$
y(z)=y_0+\Delta z\,t_y.
$$

Because this function depends only on its arguments and does not modify external state, it is a pure function. Calling it repeatedly with the same track and target depth always produces the same result.

In [8]:
from ams_ecal import project_track_to_z

first_layer_z_mm = geometry.uniform_layer_centers_z_mm[0]

first_projected_point = project_track_to_z(
    track,
    target_z_mm=first_layer_z_mm,
)

first_projected_point

(10.706254356580894, -19.592243857111676, 4.625)

## 9. Projection to all ECAL layer centers

The geometry model supplies the center depth of all 18 ECAL layers. We can apply the same projection function independently to each depth:

$$
z_1,z_2,\ldots,z_{18}.
$$

This produces an ordered collection of continuous track intersections:

$$
\left[
(x_1,y_1,z_1),
(x_2,y_2,z_2),
\ldots,
(x_{18},y_{18},z_{18})
\right].
$$

The calculation does not yet decide which transverse coordinate each layer reads or which cell contains the projected point. Those are separate detector-mapping operations for Block 2.

In [9]:
production_projected_points = tuple(
    project_track_to_z(track, target_z_mm=z_mm)
    for z_mm in geometry.uniform_layer_centers_z_mm
)

assert len(production_projected_points) == 18

for production_point, exploratory_point in zip(
    production_projected_points,
    projected_points,
    strict=True,
):
    assert production_point == pytest.approx(exploratory_point)

(
    len(production_projected_points),
    production_projected_points[0],
    production_projected_points[-1],
)

(18,
 (10.706254356580894, -19.592243857111676, 4.625),
 (34.7189024803313, -5.728534998908618, 161.875))

The reusable implementation agrees with the earlier exploratory calculation at every layer.

The first and last projected coordinates differ because the track is inclined. As the target depth increases, both transverse coordinates increase according to the constant slopes

$$
t_x=\tan\theta\cos\phi,
$$

and

$$
t_y=\tan\theta\sin\phi.
$$

The slope remains constant because this block assumes straight-line propagation. A curved trajectory would require position-dependent direction components and could not be represented by one constant pair of slopes.

## 10. Vertical-track verification using the production function

For a vertical track,

$$
\theta=0,
$$

so

$$
t_x=t_y=0.
$$

The projection function should consequently preserve the reference values $x_0$ and $y_0$ at every target depth.

In [10]:
vertical_track = TrackState(
    x0_mm=10.0,
    y0_mm=-20.0,
    z0_mm=0.0,
    theta_rad=0.0,
    phi_rad=radians(120.0),
)

production_vertical_points = tuple(
    project_track_to_z(vertical_track, target_z_mm=z_mm)
    for z_mm in geometry.uniform_layer_centers_z_mm
)

assert all(
    x_mm == pytest.approx(vertical_track.x0_mm)
    for x_mm, _, _ in production_vertical_points
)
assert all(
    y_mm == pytest.approx(vertical_track.y0_mm)
    for _, y_mm, _ in production_vertical_points
)

production_vertical_points[0], production_vertical_points[-1]

((10.0, -20.0, 4.625), (10.0, -20.0, 161.875))

## 11. Projection from an upstream reference plane

The track reference point does not have to be located at the ECAL entrance.

For example, suppose the reconstructed state is defined at

$$
z_0=-100\ \text{mm}.
$$

Projecting it to the first ECAL layer uses

$$
\Delta z=z_1-z_0.
$$

Because $z_0$ is negative, the total propagation distance in the $z$ direction is larger than the first layer's depth measured from the ECAL entrance.

In [11]:
upstream_track = TrackState(
    x0_mm=10.0,
    y0_mm=-20.0,
    z0_mm=-100.0,
    theta_rad=radians(10.0),
    phi_rad=radians(30.0),
)

upstream_first_layer_point = project_track_to_z(
    upstream_track,
    target_z_mm=first_layer_z_mm,
)

upstream_first_layer_point

(25.976618823194826, -10.775894821688427, 4.625)

## 12. Input validation

A target depth must be a finite real number. Values such as `nan` and infinity do not identify a physical detector plane and must be rejected.

The projection does not currently reject points lying outside the transverse ECAL boundaries. Such a point is still a mathematically valid intersection with the layer plane. Determining whether it corresponds to an active ECAL cell belongs to the discrete mapping developed in Block 2.

In [12]:
with pytest.raises(TypeError, match="must be a real number"):
    project_track_to_z(track, target_z_mm=True)

with pytest.raises(ValueError, match="must be finite"):
    project_track_to_z(track, target_z_mm=float("nan"))

"Projection validation checks passed."

'Projection validation checks passed.'

## Block 1 conclusions

Block 1 established a complete continuous track-projection model.

We have:

1. represented the reconstructed track with a reference position and direction angles;
2. derived the Cartesian unit direction vector;
3. derived the transverse slopes with respect to ECAL depth;
4. introduced a validated and immutable `TrackState`;
5. implemented a pure function for projection to any finite target $z$ coordinate;
6. projected inclined and vertical tracks to all 18 ECAL layer centers;
7. verified that the reusable implementation agrees with the exploratory equations;
8. tested projection from an upstream reference plane.

The final propagation equations are

$$
x(z)=x_0+(z-z_0)\tan\theta\cos\phi,
$$

and

$$
y(z)=y_0+(z-z_0)\tan\theta\sin\phi.
$$

The results remain continuous physical coordinates measured in millimetres. No cell index, readout view, or shower crop has yet been assigned.

Block 2 will connect these projected points to the alternating ECAL readout geometry and convert the appropriate transverse coordinate into a discrete cell index.